# Lab B — OpenVINO Acceleration on the AMD CPU / OpenVINO 加速 AMD CPU
## Slide 34–38 · 不换 CPU，只换推理运行时

**What will you learn? / 本节学什么？**
- 什么是推理运行时（inference runtime）与计算图优化 What an inference runtime and graph optimization are
- 在**同一块 AMD CPU** 上对比 PyTorch 与 OpenVINO 的真实延迟与 mAP Measured latency and mAP of PyTorch vs OpenVINO on the **same AMD CPU**

**One key idea / 一个核心原理**
> 同一个模型、同一块 CPU，不同的运行时可以用不同的算子实现、内存布局和线程调度——执行速度因此可能不同。Same model + same CPU, but a different runtime can > choose better operators, memory layouts and threading — so speed can differ.

In [ ]:
from pathlib import Path
import sys
REPO = Path.cwd()
if REPO.name == "notebooks":
    REPO = REPO.parent
sys.path.insert(0, str(REPO))
print("repo:", REPO)

## 1. Export the SAME weights to OpenVINO / 从同一份权重导出 OpenVINO
OpenVINO 不能直接跑 `.pt`，需要导出成 IR（xml+bin）。导出是**一次性**的，且必须来自同一份官方 `yolo26l.pt`（本模块已做缓存）。
OpenVINO needs an IR export of the same official `yolo26l.pt` (cached after the first export).

In [ ]:
import time
from course.model import export_openvino_fp32, openvino_fp32_dir, OFFICIAL_PT

ov_dir = export_openvino_fp32(640)
size_mb = sum(f.stat().st_size for f in ov_dir.iterdir()
              if f.suffix in (".xml", ".bin")) / 1e6
print("OpenVINO FP32 export:", ov_dir)
print(f"IR size: {size_mb:.1f} MB  (from the same yolo26l.pt, sha-identical weights)")

## 2. Run the baseline: PyTorch CPU / 基线：PyTorch CPU

In [ ]:
import torch, pandas as pd
from ultralytics import YOLO
from course.benchmark import bench, save_run, RunMeta, load_bench_images

print("torch threads:", torch.get_num_threads(), "(physical cores default)")
pt_model = YOLO(str(OFFICIAL_PT))
images = load_bench_images(); IMG = images[0]

WARMUP, ITERS = 3, 10
pt_ms = bench(lambda: pt_model.predict(source=str(IMG), imgsz=640, device="cpu",
                             verbose=False),
              warmup=WARMUP, iters=ITERS, sync=False)
pt_sum = save_run(RunMeta("labB_runtime_cpu", "pytorch_cpu", "cpu",
                          "pytorch-rocm-cpu", warmup=WARMUP, iters=ITERS,
                          notes="notebook quick run"), pt_ms)
print(f"PyTorch CPU P50 {pt_sum['stats']['p50_ms']:.1f} ms")

## 3. Change one thing: runtime → OpenVINO CPU / 只改一件事：换成 OpenVINO
公平性说明：同一块 CPU、同一张图片、相同的 imgsz=640 / batch=1 / FP32、相同的 ultralytics 前后处理路径（AutoBackend）。首次加载+编译时间单独记录，不计入推理延迟。
Fairness: same CPU, same image, same imgsz/batch/precision, same ultralytics pre/post path. First load+compile is timed separately and never mixed into inference latency.

In [ ]:
t0 = time.perf_counter()
ov_model = YOLO(str(ov_dir))
ov_model.predict(source=str(IMG), imgsz=640, device="cpu", verbose=False)
first_load_s = time.perf_counter() - t0
print(f"first load + graph compile: {first_load_s:.2f} s (one-time cost)")

ov_ms = bench(lambda: ov_model.predict(source=str(IMG), imgsz=640, device="cpu",
                             verbose=False),
              warmup=WARMUP, iters=ITERS, sync=False)
ov_sum = save_run(RunMeta("labB_runtime_cpu", "openvino_cpu", "cpu",
                          "openvino-cpu", warmup=WARMUP, iters=ITERS,
                          extra={"first_load_and_compile_s": round(first_load_s, 2)}),
                  ov_ms)
print(f"OpenVINO CPU P50 {ov_sum['stats']['p50_ms']:.1f} ms")

## 4. Compare: latency AND accuracy / 对比：延迟与精度一起看
运行时换了，检测质量不能悄悄变坏。我们在固定验证子集上测 mAP50-95（Educational COCO Subset，非官方 COCO 榜单）。
A runtime swap must not silently degrade quality — we measure mAP50-95 on the fixed educational val subset.

In [ ]:
from course.benchmark import map5095

pt_map = map5095(pt_model, str(REPO / "data/val_subset.yaml"), 640, 8, "cpu")
ov_map = map5095(ov_model, str(REPO / "data/val_subset.yaml"), 640, 8, "cpu")
sp = pt_sum["stats"]["p50_ms"] / ov_sum["stats"]["p50_ms"]
display(pd.DataFrame([
    {"runtime": "PyTorch CPU", "P50 (ms)": round(pt_sum['stats']['p50_ms'], 1),
     "speedup": "1.00x", "mAP50-95": round(pt_map['map50_95'], 4)},
    {"runtime": "OpenVINO CPU", "P50 (ms)": round(ov_sum['stats']['p50_ms'], 1),
     "speedup": f"{sp:.2f}x", "mAP50-95": round(ov_map['map50_95'], 4)}]))

In [ ]:
from course.charts import chart_pytorch_vs_openvino
chart_pytorch_vs_openvino(pt_sum, ov_sum, pt_map["map50_95"], ov_map["map50_95"])

## Common mistakes / 常见错误
- 认为“导出即加速”：导出只是换格式，加速来自编译期的图优化与算子选择，且**必须实测验证**。
  *Export alone accelerates nothing* — speedup comes from graph optimization at compile time, and must be measured.
- 混淆首次编译时间与推理延迟：编译只发生一次，不应计入稳态延迟。
  *First compile is one-time cost* — never mix it into steady-state latency.
- 宣称 AMD CPU 有 AMX：**没有**。AMD Zen4 的 INT8 路径是 AVX-512 VNNI（见 Lab C）。
  *Never claim AMD CPUs have Intel AMX* — they do not; AMD's INT8 path is AVX-512 VNNI (see Lab C).


## 5. Why? / 为什么会这样？
OpenVINO 导出时会对计算图做算子融合、内存布局重排、常量折叠，并用 oneDNN 生成针对本 CPU 指令集（AVX-512 / AVX-512 VNNI，**AMD Zen4 没有 Intel AMX**）的内核。PyTorch 也有优化，但它是为**训练**设计的通用框架。
OpenVINO performs operator fusion, layout selection and constant folding at export, then generates oneDNN kernels for this CPU's ISA (AVX-512 / AVX-512 VNNI — **AMD Zen4 has NO Intel AMX**). PyTorch is a general training-first framework.

**重要 / Important**: OpenVINO 官方对 AMD CPU 的验证范围有限；本实验以本机实测为准（无论结果是更快、持平还是更慢，数据说了算）。
OpenVINO's official validation on AMD CPUs is limited; on this machine the measurement is the authority — faster, equal or slower, the data decides.


## 6. Try it yourself — see the runtime with your own eyes / 学生练习：亲眼看看运行时

**先预测**：把 `STUDENT_HINT` 从 `"LATENCY"` 改成 `"THROUGHPUT"`，单张延迟会变快还是变慢？
**Predict first**: changing `STUDENT_HINT` from `"LATENCY"` to `"THROUGHPUT"` — will single-image latency go up or down?

下面**不用 ultralytics**，直接用 OpenVINO Runtime 五步走：读取 IR → `ov.Core()` → 选择 CPU Plugin → 编译 → 推理。
你想看到的是：**不是换了一行代码就神奇加速，而是同一个模型走了不同的执行引擎**。
Below we bypass ultralytics and use the OpenVINO runtime directly — read IR → `ov.Core()` → CPU plugin → compile → infer.
The point: **it is not a magic line of code — the same model now runs through a different execution engine.**

（这是**额外探索**：THROUGHPUT 模式面向批量吞吐，用更多并行流；LATENCY 面向单请求。它不影响上面的公平对比。）
(Extra exploration: THROUGHPUT mode targets batch throughput with more streams; LATENCY targets single requests. It does not affect the fair comparison above.)


In [ ]:
STUDENT_HINT = "LATENCY"       # <-- change me: "LATENCY" or "THROUGHPUT"
my_hint_prediction = "single-image latency will get ___"   # <-- 写你的预测

import numpy as np, time
import openvino as ov

# 5 real steps of the OpenVINO runtime / OpenVINO 运行时五步走
core = ov.Core()                                  # 1. runtime instance
ov_model = core.read_model(str(ov_dir / "yolo26l.xml"),
                           str(ov_dir / "yolo26l.bin"))   # 2. read the IR
compiled = core.compile_model(                    # 3. CPU plugin + compile
    ov_model, "CPU", {"PERFORMANCE_HINT": STUDENT_HINT})
ireq = compiled.create_infer_request()            # 4. infer request
x = np.random.rand(1, 3, 640, 640).astype(np.float32)
ireq.infer(x)                                     # 5. infer (warm-up)

t = []
for _ in range(10):
    t0 = time.perf_counter(); ireq.infer(x); t.append((time.perf_counter()-t0)*1000)
t.sort()
print(f"hint={STUDENT_HINT} · model-only forward P50: {t[len(t)//2]:.1f} ms")
print("your prediction:", my_hint_prediction)


**Takeaway / 一句话总结**
> 模型和 CPU 都不变，只换推理运行时，速度也可能明显变化——因为“怎么算”和“算什么”一样重要。
> With the same model and CPU, changing only the runtime can change speed > — *how* you compute matters as much as *what* you compute.